# Milestone 1 - Explanatory Info

Milestone 1 requires you to document your code and offer extended explanations using **markdown cells**. 

To create and edit markdown cells:
- click inside the cell
- make sure *Markdown* is selected in the cell type dropdown menu
- Write text in the cell
- **Run the cell** to render the markdown
- Double-click in a markdown cell to edit them (double-click on this cell to see how this works)

## Workspace

We'll work in our group project directories for this project. These will not be set up on day 1, but should be available shortly after the groups are selected. Your group project directory will look something like: `/courses/meteo473/sp26/group0/` (replace `0` with your group number). We will create a symbolic link from your home directory to this location, so you can access it.

Some things to keep in mind:
- Make sure your group project files, especially your data, are in the group project directory, *not* your home directory. If you have questions, please ask!
- **Manage your data wisely**. Periodically clean up unnecessary files (and avoid unnecessary downloads in the first place).
- You will *not* want to edit the same file at the same time as a groupmate. This can cause unexpected behavior, so the best practice is to work in separate notebooks until you're ready to put the pieces together.
- We may encounter file permission errors, so please be patient and notify me of issues as soon as they occur.

## Model Data Download using Herbie
Herbie is a python library developed by Brian Blaylock, and one that I (Karl) have contributed to! It's designed to simplify the process of downloading model data and reading it into xarray.

Documentation: https://herbie.readthedocs.io/en/stable/

GitHub: https://github.com/blaylockbk/Herbie

In [1]:
from herbie import Herbie, FastHerbie
import pandas as pd, numpy as np
import xarray as xr
import dask

 ╭─▌▌Herbie─────────────────────────────────────────────╮
 │ INFO: Created a default config file.                 │
 │ You may view/edit Herbie's configuration here:       │
 │      /home/jhk5405/.config/herbie/config.toml        │
 ╰──────────────────────────────────────────────────────╯



In [2]:
# 0z today
run = pd.Timestamp("now", tz="utc").replace(tzinfo=None).floor('24h')

In [11]:
run = pd.Timestamp("2026-03-16-00")

### ECMWF data download example
**Step 1: Create the Herbie object**

The Herbie object lets us investigate and download remote forecast model data.
- `run` is the valid time of the model run we want to grab (datetime-like)
- `model` is the model to grab, in this case "ifs" (ecmwf)
- `product` is required for this particular model. "oper" tells us it's the operational model, not the ensemble
- `fxx` is the forecast hour
- `save_dir` tells Herbie where to store the downloads. This puts it in a folder called data (in the same directory as this notebook)
- `overwrite` tells Herbie to overwrite data in the download location, even if it has been downloaded previously

**Important Note:** you'll want to set `save_dir` to a data folder within your group project directory when you begin the milestone work to avoid cluttering your home directory (e.g., `/courses/meteo473/sp26/group0/data/`).

In [12]:
run

Timestamp('2026-03-16 00:00:00')

In [13]:
H = Herbie(run, model="ifs", product="oper", fxx=6, save_dir='./data/', overwrite=True)

✅ Found ┊ model=ifs ┊ product=oper ┊ 2026-Mar-16 00:00 UTC F06 ┊ GRIB2 @ google ┊ IDX @ google


**Step 2: Determine which parameters to download**

Model data is often stored as grib2 files. These are binary files containing multiple "grib messages". Each grib message corresponds to a particular parameter. Often, we are only interested in just a handful of parameters, so instead of downloading the whole file, we can download a subset. The `inventory()` method of the Herbie object shows the contents of the remote grib file:

In [14]:
H.inventory()

,grib_message,start_byte,end_byte,range,reference_time,valid_time,step,param,levelist,levtype,number,domain,expver,class,type,stream,search_this
0,1,0,187781,0-187781,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,lsm,NaN,sfc,NaN,g,0001,od,fc,oper,:lsm:sfc:g:0001:od:fc:oper
1,2,187781,298647,187781-298647,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,asn,NaN,sfc,NaN,g,0001,od,fc,oper,:asn:sfc:g:0001:od:fc:oper
2,3,298647,762808,298647-762808,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,q,150,pl,NaN,g,0001,od,fc,oper,:q:150:pl:g:0001:od:fc:oper
3,4,762808,1425552,762808-1425552,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,tcwv,NaN,sfc,NaN,g,0001,od,fc,oper,:tcwv:sfc:g:0001:od:fc:oper
4,5,1425552,1714327,1425552-1714327,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,rsn,NaN,sfc,NaN,g,0001,od,fc,oper,:rsn:sfc:g:0001:od:fc:oper
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,157,124848664,125390983,124848664-125390983,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,sp,NaN,sfc,NaN,g,0001,od,fc,oper,:sp:sfc:g:0001:od:fc:oper
157,158,125390983,125818961,125390983-125818961,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,gh,150,pl,NaN,g,0001,od,fc,oper,:gh:150:pl:g:0001:od:fc:oper
158,159,125818961,126489593,125818961-126489593,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,tcw,NaN,sfc,NaN,g,0001,od,fc,oper,:tcw:sfc:g:0001:od:fc:oper
159,160,126489593,127846231,126489593-127846231,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,d,400,pl,NaN,g,0001,od,fc,oper,:d:400:pl:g:0001:od:fc:oper


You may use search strings, passed as keyword arguments to the `inventory()` and `download()` methods, to get a subset of parameters:

In [17]:
# This is actually a regular expression. Search "regular expressions" for examples
ss = r":(2t|2d):"

In [18]:
# 2m temp
H.inventory(ss)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


,grib_message,start_byte,end_byte,range,reference_time,valid_time,step,param,levelist,levtype,number,domain,expver,class,type,stream,search_this
93,94,72144863,72808664,72144863-72808664,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,2t,NaN,sfc,NaN,g,0001,od,fc,oper,:2t:sfc:g:0001:od:fc:oper
100,101,77810081,78498103,77810081-78498103,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,2d,NaN,sfc,NaN,g,0001,od,fc,oper,:2d:sfc:g:0001:od:fc:oper


In [13]:
# search strings can be complicated regular expressions to match multiple parameters at once
# height, temperature, and wind on mandatory levels
ss2 = "((gh|t|u|v):(1000|850|700|500|250))"
H.inventory(ss2)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


,grib_message,start_byte,end_byte,range,reference_time,valid_time,step,param,levelist,levtype,number,domain,expver,class,type,stream,search_this
19,20,12628936,13332973,12628936-13332973,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,u,500,pl,NaN,g,0001,od,fc,oper,:u:500:pl:g:0001:od:fc:oper
21,22,14688569,15423461,14688569-15423461,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,v,500,pl,NaN,g,0001,od,fc,oper,:v:500:pl:g:0001:od:fc:oper
26,27,18199052,18692538,18199052-18692538,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,gh,850,pl,NaN,g,0001,od,fc,oper,:gh:850:pl:g:0001:od:fc:oper
32,33,22854335,23444872,22854335-23444872,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,t,850,pl,NaN,g,0001,od,fc,oper,:t:850:pl:g:0001:od:fc:oper
51,52,34767373,35491044,34767373-35491044,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,u,250,pl,NaN,g,0001,od,fc,oper,:u:250:pl:g:0001:od:fc:oper
54,55,37116519,37749301,37116519-37749301,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,v,250,pl,NaN,g,0001,od,fc,oper,:v:250:pl:g:0001:od:fc:oper
87,88,66788350,67511780,66788350-67511780,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,u,850,pl,NaN,g,0001,od,fc,oper,:u:850:pl:g:0001:od:fc:oper
92,93,71398620,72144863,71398620-72144863,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,v,850,pl,NaN,g,0001,od,fc,oper,:v:850:pl:g:0001:od:fc:oper
103,104,80382919,80933190,80382919-80933190,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,gh,1000,pl,NaN,g,0001,od,fc,oper,:gh:1000:pl:g:0001:od:fc:oper
105,106,81807634,82294414,81807634-82294414,2026-03-16,2026-03-16 06:00:00,0 days 06:00:00,gh,700,pl,NaN,g,0001,od,fc,oper,:gh:700:pl:g:0001:od:fc:oper


**Step 3: Download the data**

We'll then use the `download()` method of the Herbie object to download and save the data as a grib2 file. Remember to pass the search string as an argument in order to ensure that only the desired parameters are saved.

In [19]:
fp = H.download(ss)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


**Step 4: Read in data using xarray**

We can use xarray to read in grib2 files using the `cfgrib` backend.

In [15]:
fp

PosixPath('data/ifs/20260316/subset_3fb2366e__20260316000000-6h-oper-fc.grib2')

In [16]:
ds = xr.open_dataset(fp)
ds

<xarray.Dataset> Size: 4MB
Dimensions:            (latitude: 721, longitude: 1440)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    t2m                (latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-16T14:27 GRIB to CDM+CF via cfgrib-0.9.1...

**IMPORTANT NOTE**

Now that we have downloaded and saved our data, there is no need to re-download it again. You may save the file path for future reference and use `xr.open_dataset()` to load in your data each time you come back to work in your notebook. 

Alternatively, you may want to save your dataset as a more recognizable name (the value associated with `fp` is too long to remember). We'll export the dataset as a different file type called netCDF.

In [17]:
filename = 'ecmwf_t2m_006.nc'

In [18]:
ds.to_netcdf(filename)

In [19]:
# Now, when you come back to this notebook, simply run:
ds = xr.open_dataset(filename)
ds

sh: 1: getfattr: not found


<xarray.Dataset> Size: 4MB
Dimensions:            (latitude: 721, longitude: 1440)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    t2m                (latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-16T14:27 GRIB to CDM+CF via cfgrib-0.9.1...

## Downloading an entire run
Using the FastHerbie class allows you to download multiple runs and/or forecast hours in parallel.

**Note:** The assignment guidelines ask you to download a *complete model run*. For the purposes of this milestone, you should do the following:
- choose a run time that is slightly before the beginning of your weather event
- make sure you download enough forecast time steps to capture the duration of the entire event, as well as several null forecast times (no event happening)
- you don't have to download all time steps out to 240h, but you should have enough to capture both the event of interest and null times
- save all of your data in one file; do not have separate files for each time step

In [23]:
H = FastHerbie([run], model="ifs", product="oper", fxx=np.arange(0,90,6).tolist(), save_dir='./data/', overwrite=True)

In [24]:
filepath = H.download(ss)

In [25]:
filepath

[PosixPath('data/ifs/20260316/subset_3ff1366e__20260316000000-72h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3fef90f4__20260316000000-0h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f1c48ac__20260316000000-48h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f8948ac__20260316000000-18h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f78b6ae__20260316000000-54h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f0ea8c6__20260316000000-24h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3fe714f3__20260316000000-42h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f607a5a__20260316000000-30h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f052f9a__20260316000000-84h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f0e7441__20260316000000-36h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3fb2366e__20260316000000-6h-oper-fc.grib2'),
 PosixPath('data/ifs/20260316/subset_3f1248ac__20260316000000-12h-oper-fc.grib2'),
 Posix

**Opening multiple files at once**

Because we downloaded multiple files (multiple parameters + multiple forecast hours), filepath is a list of paths, not a single file path. To open multiple files at once, we use xarray's `open_mfdataset()` method, with expects a list of file paths:

In [38]:
ds = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time')

Ignoring index file '/home/kps5442/courses/meteo473/sp26/lessons/data/ifs/20260316/subset_3fb2366e__20260316000000-6h-oper-fc.grib2.5b7b6.idx' older than GRIB file
/tmp/ipykernel_60687/820905488.py:1: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time')


In [39]:
ds

<xarray.Dataset> Size: 62MB
Dimensions:            (valid_time: 15, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 120B 2026-03-19 ... 2026-0...
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    time               datetime64[ns] 8B 2026-03-16
    step               (valid_time) timedelta64[ns] 120B 3 days 00:00:00 ... ...
    heightAboveGround  float64 8B 2.0
Data variables:
    t2m                (valid_time, latitude, longitude) float32 62MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-16T14:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [40]:
# May be necessary to ensure data is in the correct order
ds = ds.sortby('valid_time')
ds

<xarray.Dataset> Size: 62MB
Dimensions:            (valid_time: 15, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 120B 2026-03-16 ... 2026-0...
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    time               datetime64[ns] 8B 2026-03-16
    step               (valid_time) timedelta64[ns] 120B 0 days 00:00:00 ... ...
    heightAboveGround  float64 8B 2.0
Data variables:
    t2m                (valid_time, latitude, longitude) float32 62MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-16T14:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [41]:
# Always clip your data to area of interest before saving
ds = ds.sel(latitude=slice(60,20), longitude=slice(-130,-60))
ds

<xarray.Dataset> Size: 3MB
Dimensions:            (valid_time: 15, latitude: 161, longitude: 281)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 120B 2026-03-16 ... 2026-0...
  * latitude           (latitude) float64 1kB 60.0 59.75 59.5 ... 20.25 20.0
  * longitude          (longitude) float64 2kB -130.0 -129.8 ... -60.25 -60.0
    time               datetime64[ns] 8B 2026-03-16
    step               (valid_time) timedelta64[ns] 120B 0 days 00:00:00 ... ...
    heightAboveGround  float64 8B 2.0
Data variables:
    t2m                (valid_time, latitude, longitude) float32 3MB dask.array<chunksize=(1, 161, 281), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-16T14:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [43]:
# Save as a netcdf file
# If the file already exists, delete it before tying again, otherwise you will see an error
fname = f'ecmwf_{run:%Y%m%d%H}.nc'
ds.to_netcdf(fname)

In [44]:
# Should be the same since we are opening the file we just created
ds_test = xr.open_dataset(fname)
ds_test

sh: 1: getfattr: not found


<xarray.Dataset> Size: 3MB
Dimensions:            (valid_time: 15, latitude: 161, longitude: 281)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 120B 2026-03-16 ... 2026-0...
  * latitude           (latitude) float64 1kB 60.0 59.75 59.5 ... 20.25 20.0
  * longitude          (longitude) float64 2kB -130.0 -129.8 ... -60.25 -60.0
    time               datetime64[ns] 8B ...
    step               (valid_time) timedelta64[ns] 120B ...
    heightAboveGround  float64 8B ...
Data variables:
    t2m                (valid_time, latitude, longitude) float32 3MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-16T14:35 GRIB to CDM+CF via cfgrib-0.9.1...